In [ ]:
%run src/donnees.py

# df_caract_recoder
# df_lieux_recoder
# df_vehicule_recoder
# df_usager_recoder
# df_final

Problématique : Quels facteurs influencent la gravité des accidents corporels de la route en France, peut-on prédire cette gravité en tenant compte des caractéristiques de l'accident (environnementales, géographique...) ?

# Cartographie

Nous souhaitons voir si certaines régions (ou departements) ont plus d'accidents grave ... ?
Par accident, il peut y avoir plusiuers victime, nous considérons donc le nombre de victime et non le nombre d'accident 

In [ ]:
# nombre d'accident
df_final["Num_Acc"].nunique()

In [ ]:
# nombre de victime 
df_final["id_usager"].nunique()

In [ ]:
# recupération du nombre de victime par dep et grav  (utilisation longitute lattiude ?)
df_victime_dep = df_final.groupby(["dep", "grav"]).size().reset_index(name="nb")

In [ ]:
df_victime_dep

In [ ]:
df_final["grav"].unique()
# suppression des nan - automatique avec le groupby fait

In [ ]:
df_victime_tue = df_victime_dep[df_victime_dep["grav"] == "Tué"]

In [ ]:
df_victime_tue

Carte des accidents grave puis carte avec chaque type d'accident
toute année confondu 

In [ ]:
import matplotlib.pyplot as plt
!pip install cartiflette
from cartiflette import carti_download

def initialisation_carte():
    departement_borders = carti_download(
        values=["France"],
        crs=4326,
        borders="DEPARTEMENT",
        vectorfile_format="geojson",
        simplification=50,
        filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
        source="EXPRESS-COG-CARTO-TERRITOIRE",
        year=2022)

    departement_borders = departement_borders.rename(columns={"INSEE_DEP": "dep"})
    return departement_borders




def carte(blessure, df):
    """
    Fonction pour créer une carte par blessure
    """
    if blessure == "non":
        df_blessure = df.groupby(["dep"]).size().reset_index(name="nb")
    else:
        df_blessure = df[df["grav"] == blessure]

    carte = initialisation_carte().merge(
        df_blessure,
        on="dep",
        how="left"
    )

    # Carte
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    carte.plot(
            column="nb",
            cmap="RdBu_r",
            legend=True,
            edgecolor="black",
            linewidth=0.5,
            ax=ax
        )

    ax.set_title(f"Nombre de {blessure} par département sur toute les années", fontsize=16)
    ax.axis("off")

    plt.show()

    

Une carte sur le nombre d'accident pour voir la répartition 
Une carte toute victime confondu ?

In [ ]:
carte("Tué",df_victime_dep)

In [ ]:
carte("Blessé hospitalisé",df_victime_dep)

In [ ]:
carte("Indemne",df_victime_dep)

In [ ]:
carte("Blessé léger",df_victime_dep)

In [ ]:
carte("non",df_final)

Il y a bcp d'accident en ile de france, interressantde plutot faire un pourcentage des accidnets par region 

In [ ]:
df_victime_tot = df_final.groupby(["dep"]).size().reset_index(name="nb").sort_values(by="nb")

In [ ]:
df_victime_tot

Proportion d'une catégorie de blessure dans le nb de victime total par dep
(faire nb d'accidnet ? en mode un mortel = au moins une victime ?)

In [ ]:
# enlever les na ????
df_victimes_total = (
    df_final
    .dropna(subset=["dep", "grav"])
    .groupby("dep")
    .size()
    .reset_index(name="nb_victimes")
)

df_victime_dep2 = df_final.groupby(["dep", "grav"]).size().reset_index(name="nb")

df_dep = df_victime_dep2.merge(df_victimes_total, on="dep", how="left")
df_dep["proportion"] = df_dep["nb"] / df_dep["nb_victimes"] *100

In [ ]:
df_dep[df_dep["grav"] == "Tué"].sort_values(by="proportion")


In [ ]:
import matplotlib.ticker as mtick
def carte2(blessure, df):
    """
    Fonction pour créer une carte par blessure proportion par dep
    """

    df_blessure = df[df["grav"] == blessure]

    carte = initialisation_carte().merge(
        df_blessure,
        on="dep",
        how="left"
    )

    # Carte
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    m=carte.plot(
            column="proportion",
            cmap="RdBu_r",
            legend=True,
            edgecolor="black",
            linewidth=0.5,
            ax=ax
        )

        # Récupérer la colorbar
    cbar = m.get_figure().axes[-1]

    # Formater en pourcentage
    cbar.yaxis.set_major_formatter(mtick.PercentFormatter())


    ax.set_title(f"Proportion de {blessure} par département sur toute les années", fontsize=16)
    ax.axis("off")

    plt.show()


    

In [ ]:
carte2("Tué", df_dep)

In [ ]:
df_dep.sort_values(by="proportion")

In [ ]:
carte2("Indemne", df_dep)

Les victimes tués par un accdient sont une faible proportion des victimes, ou on en voit le plus dans la diagonal du vide 
En revanche, les accidents indemne sont plus vers le nord 
comparaison année à faire 